# Обучение модели ResNet18 для классификации AI vs Human Generated Images

Этот notebook содержит код для обучения модели ResNet18 на датасете Train_1 и оценки качества на Test_1.

## 1. Импорт библиотек

In [1]:
# pip install torch==2.8.0 torchvision==0.23.0

In [2]:
import torch
import boto3
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms, models
from PIL import Image
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score
from tqdm import tqdm
import matplotlib.pyplot as plt

device = torch.device('cuda:3' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

Using device: cuda:3


## 2. Подготовка данных

In [3]:
class ImageDataset(Dataset):
    def __init__(self, csv_file, root_dir, transform=None):
        self.data = pd.read_csv(csv_file)
        self.root_dir = Path(root_dir)
        self.transform = transform
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        img_path = self.root_dir / self.data.iloc[idx]['file_name']
        image = Image.open(img_path).convert('RGB')
        label = self.data.iloc[idx]['label']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [4]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset/Train_1/train.csv',
    root_dir='ai-vs-human-generated-dataset/Train_1',
    transform=train_transform
)

test_dataset = ImageDataset(
    csv_file='ai-vs-human-generated-dataset/Test_1/test.csv',
    root_dir='ai-vs-human-generated-dataset/Test_1',
    transform=test_transform
)

batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

print(f'Train dataset size: {len(train_dataset)}')
print(f'Test dataset size: {len(test_dataset)}')

Train dataset size: 9993
Test dataset size: 3997


## 3. Создание модели ResNet18

In [5]:
model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, 2)
model = model.to(device)

print(f'Model architecture:')
print(model)

/data/shared_ml/pyudakov/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/data/shared_ml/pyudakov/.venv/lib/python3.10/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Model architecture:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): R

## 4. Определение функции потерь и оптимизатора

In [6]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

## 5. Функция обучения

In [7]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    total_samples = 0
    
    for images, labels in tqdm(dataloader, desc='Training'):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        batch_size = images.size(0)
        total_samples += batch_size
        running_loss += loss.item() * batch_size

        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, epoch_f1

## 6. Функция валидации

In [8]:
def validate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds = []
    all_labels = []
    total_samples = 0
    
    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc='Validation'):
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            batch_size = images.size(0)
            total_samples += batch_size
            running_loss += loss.item() * batch_size

            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total_samples
    epoch_acc = accuracy_score(all_labels, all_preds)
    epoch_f1 = f1_score(all_labels, all_preds)
    epoch_precision = precision_score(all_labels, all_preds)
    epoch_recall = recall_score(all_labels, all_preds)
    
    return epoch_loss, epoch_acc, epoch_f1, epoch_precision, epoch_recall

## 7. Обучение модели

In [ ]:
from botocore.client import Config

def upload_to_minio(file_path, bucket_name, object_name):
    s3 = boto3.client(
        's3',
        endpoint_url='http://localhost:6062',
        aws_access_key_id='minioadmin',
        aws_secret_access_key='minioadmin',
        region_name='us-east-1',
        config=Config(signature_version='s3v4')
    )
    try:
        s3.head_bucket(Bucket=bucket_name)
    except Exception:
        s3.create_bucket(Bucket=bucket_name)

    s3.upload_file(file_path, bucket_name, object_name)
    print(f"Загружено: s3://{bucket_name}/{object_name}")

In [ ]:
exp_name = 'exp1'
writer = SummaryWriter(log_dir=f'logs/runs/{exp_name}')

hparams = {
    'learning_rate': 0.001,
    'batch_size': 32,
    'epochs': 10,
    'optimizer': 'Adam',
    'model': 'ResNet18',
    'scheduler': 'StepLR',
    'step_size': 0.5,
    'gamma': 0.1,
    'dataset': 'CIFAR10'
}

params_text = "\n".join([f"{k}: {v}" for k, v in hparams.items()])
writer.add_text('hyperparameters_formatted', params_text, 0)

num_epochs = 10
train_losses = []
train_accs = []
train_f1s = []

torch.backends.cudnn.enabled = False

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 50)
    
    train_loss, train_acc, train_f1 = train_epoch(model, train_loader, criterion, optimizer, device)
    scheduler.step()
    
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    train_f1s.append(train_f1)

    writer.add_scalar('Loss/train', train_loss, epoch)
    writer.add_scalar('Accuracy/train', train_acc, epoch)
    writer.add_scalar('F1/train', train_f1, epoch)
    
    print(f'Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, F1: {train_f1:.4f}')

print('\nTraining completed!')

model_dir = Path("logs/artifacts") / exp_name
model_dir.mkdir(parents=True, exist_ok=True)
model_path = model_dir / "resnet18_final.pth"
torch.save(model.state_dict(), str(model_path))

upload_to_minio(
    file_path=str(model_path),
    bucket_name="models",
    object_name=f"{exp_name}/resnet18_final.pth"
)



Epoch 1/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:26<00:00, 11.97it/s]


Train Loss: 0.3536, Acc: 0.8527, F1: 0.8525

Epoch 2/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:26<00:00, 11.89it/s]


Train Loss: 0.2682, Acc: 0.8909, F1: 0.8912

Epoch 3/10
--------------------------------------------------


...

Train Loss: 0.0859, Acc: 0.9680, F1: 0.9680

Epoch 9/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:26<00:00, 11.92it/s]


Train Loss: 0.0663, Acc: 0.9728, F1: 0.9728

Epoch 10/10
--------------------------------------------------


Training: 100%|██████████| 313/313 [00:26<00:00, 11.90it/s]


Train Loss: 0.0646, Acc: 0.9750, F1: 0.9750

Training completed!
Загружено: s3://models/exp1/resnet18_final.pth


## 8. Оценка модели на тестовом датасете

In [ ]:
# Метрики обучения и теста логируются в TensorBoard:
# tensorboard --logdir=logs/runs

In [17]:
def log_test_metrics(model, test_loader, criterion, device):
    test_loss, test_acc, test_f1, test_precision, test_recall = validate(model, test_loader, criterion, device)

    writer.add_scalar('Loss/test', test_loss, num_epochs)
    writer.add_scalar('Accuracy/test', test_acc, num_epochs)
    writer.add_scalar('F1/test', test_f1, epoch, num_epochs)
    writer.add_scalar('Precision/test', test_precision, num_epochs)
    writer.add_scalar('Recall/test', test_recall, num_epochs)
    writer.close()

In [16]:
log_test_metrics(model, test_loader, criterion, device)

Validation: 100%|██████████| 125/125 [00:05<00:00, 21.73it/s]


## 9. DVC-пайплайн и выводы

Обучение и дообучение вынесены в `train.py` и `continue_train.py`, пайплайн описан в `dvc.yaml` и запускается командой `dvc repro`.

- Датасет версионируется DVC и хранится в S3 (MinIO), каждая обученная модель выгружается туда же.
- DVC строит DAG стадий и при изменении данных, кода или параметров перезапускает только затронутые стадии, поэтому цикл переобучения не требует ручного запуска скриптов.